# MochiVoice — GPT-SoVITS v2 training (v9)

This notebook is a single audited production run for the private datasets:

- `edriannieves/mochi-train-en-v2` — 251 English training clips and `train.list`.
- `edriannieves/mochi-ref-voice` — Mochikosan Japanese reference audio.

The source repository and Hugging Face model snapshot are pinned. Temporary source,
caches, preprocessing data, and full optimizer checkpoints live under `/kaggle/tmp`;
only logs and the final six-file model package are written under `/kaggle/working`.

In [ ]:
# 1. Reproducible setup. Keep the known-good install order; pin source and model revisions.
import hashlib
import json
import os
import pathlib
import signal
import subprocess
import sys
import time

WORK = "/kaggle/working"
TMP = "/kaggle/tmp/mochi-gpt-sovits-v9"
REPO = f"{TMP}/GPT-SoVITS"
GSV_ROOT = f"{REPO}/GPT_SoVITS"
CONSTRAINTS = f"{TMP}/constraints.txt"
RUN_LOGS = f"{WORK}/run_logs"
UPSTREAM_COMMIT = "48b1a0169a28582a8984402f82cf438d3bfa6aca"
HF_REVISION = "336b2ec4e8d4ac74740798dd40af44e74659ecaf"
os.makedirs(WORK, exist_ok=True)
os.makedirs(TMP, exist_ok=True)
os.makedirs(RUN_LOGS, exist_ok=True)

def _display(cmd):
    return " ".join(str(x) for x in cmd)

def _kill_tree(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(2)
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

def stream(cmd, timeout=1800):
    cmd = [str(x) for x in cmd]
    print(f"$ {_display(cmd)}", flush=True)
    started = time.time()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )
    tail = []
    try:
        for line in proc.stdout:
            tail.append(line.rstrip())
            tail = tail[-40:]
            print(line.rstrip()[:240], flush=True)
            if time.time() - started > timeout:
                raise TimeoutError(f"timeout after {timeout}s: {_display(cmd)}")
        rc = proc.wait()
    except BaseException:
        _kill_tree(proc)
        raise
    if rc != 0:
        raise RuntimeError(f"command failed ({rc}): {_display(cmd)}\n" + "\n".join(tail))
    print(f"[exit=0 in {(time.time() - started) / 60:.1f} min]", flush=True)

pathlib.Path(CONSTRAINTS).write_text(
    "torch==2.4.1\n"
    "torchvision==0.19.1\n"
    "torchaudio==2.4.1\n"
    "x_transformers==1.42.7\n"
    "rotary_embedding_torch==0.8.9\n",
    encoding="utf-8",
)
if os.path.exists(REPO):
    import shutil
    shutil.rmtree(REPO)
stream(["git", "init", REPO], timeout=120)
stream(["git", "-C", REPO, "remote", "add", "origin", "https://github.com/RVC-Boss/GPT-SoVITS.git"], timeout=120)
stream(["git", "-C", REPO, "fetch", "--depth", "1", "origin", UPSTREAM_COMMIT], timeout=600)
stream(["git", "-C", REPO, "checkout", "--detach", UPSTREAM_COMMIT], timeout=120)
actual_commit = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == UPSTREAM_COMMIT, (actual_commit, UPSTREAM_COMMIT)
os.chdir(REPO)

py = sys.executable
stream([py, "-m", "pip", "install", "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1", "--index-url", "https://download.pytorch.org/whl/cu121"], timeout=1500)
stream([py, "-m", "pip", "install", "opencc==1.4.2"], timeout=300)
stream([py, "-m", "pip", "install", "-r", f"{REPO}/requirements.txt", "-c", CONSTRAINTS, "--progress-bar", "off"], timeout=2700)
stream(["apt-get", "install", "-y", "-q", "ffmpeg"], timeout=300)
stream([py, "-c", "import nltk; nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('punkt_tab', quiet=True)"], timeout=120)

import torch
assert torch.cuda.is_available(), "CUDA is unavailable"
capability = torch.cuda.get_device_capability(0)
assert capability == (6, 0), f"expected Tesla P100 SM 6.0, got {capability}"
print(f"SETUP DONE: {torch.cuda.get_device_name(0)} | torch {torch.__version__} | sm_{capability[0]}{capability[1]}", flush=True)

In [ ]:
# 2. Download exactly the audited pretrained snapshot.
from huggingface_hub import snapshot_download
import os

PRETRAINED = f"{GSV_ROOT}/pretrained_models"
os.makedirs(PRETRAINED, exist_ok=True)
snapshot_download(
    "lj1995/GPT-SoVITS",
    revision=HF_REVISION,
    local_dir=PRETRAINED,
    cache_dir=f"{TMP}/hf-cache",
    allow_patterns=[
        "gsv-v2final-pretrained/*",
        "chinese-roberta-wwm-ext-large/*",
        "chinese-hubert-base/*",
    ],
)
required = [
    f"{PRETRAINED}/gsv-v2final-pretrained/s2G2333k.pth",
    f"{PRETRAINED}/gsv-v2final-pretrained/s2D2333k.pth",
    f"{PRETRAINED}/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt",
    f"{PRETRAINED}/chinese-roberta-wwm-ext-large/config.json",
    f"{PRETRAINED}/chinese-roberta-wwm-ext-large/tokenizer.json",
    f"{PRETRAINED}/chinese-roberta-wwm-ext-large/pytorch_model.bin",
    f"{PRETRAINED}/chinese-hubert-base/config.json",
    f"{PRETRAINED}/chinese-hubert-base/preprocessor_config.json",
    f"{PRETRAINED}/chinese-hubert-base/pytorch_model.bin",
]
for path in required:
    assert os.path.exists(path), f"missing pretrained file: {path}"
print(f"pretrained snapshot OK: {HF_REVISION}", flush=True)

In [ ]:
# 3. Stage and validate data; define the shared streaming subprocess runner.
import glob
import os
import pathlib
import queue
import shutil
import subprocess
import sys
import threading
import wave
import zipfile

os.environ["PYTHONPATH"] = os.pathsep.join([GSV_ROOT, REPO, os.environ.get("PYTHONPATH", "")])
os.environ["PYTHONUNBUFFERED"] = "1"
DATA = f"{REPO}/Data/mochi"
EXP = "mochi"
OPT = f"{REPO}/logs/{EXP}"
os.makedirs(DATA, exist_ok=True)
os.makedirs(OPT, exist_ok=True)

def _terminate_group(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        time.sleep(2)
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

def run_logged(stage, args, extra_env=None, timeout=1800):
    args = [str(x) for x in args]
    log_path = pathlib.Path(RUN_LOGS) / f"{stage}.log"
    merged = os.environ.copy()
    merged.update(extra_env or {})
    merged["PYTHONPATH"] = os.environ["PYTHONPATH"]
    merged["PYTHONUNBUFFERED"] = "1"
    print(f"$ {' '.join(args)}", flush=True)
    proc = subprocess.Popen(
        args,
        cwd=REPO,
        env=merged,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )
    lines = queue.Queue()
    def pump():
        with log_path.open("w", encoding="utf-8", errors="replace") as log:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                lines.put(line)
        lines.put(None)
    threading.Thread(target=pump, daemon=True).start()
    started = time.time()
    tail = []
    try:
        while True:
            if time.time() - started > timeout:
                raise TimeoutError(f"{stage} exceeded {timeout}s; log={log_path}")
            try:
                line = lines.get(timeout=1)
            except queue.Empty:
                continue
            if line is None:
                break
            tail.append(line.rstrip())
            tail = tail[-30:]
            print(line.rstrip()[:240], flush=True)
        rc = proc.wait()
    except BaseException:
        _terminate_group(proc)
        raise
    if rc != 0:
        raise RuntimeError(f"{stage} failed with exit {rc}; log={log_path}\n" + "\n".join(tail))
    print(f"{stage} complete in {(time.time() - started) / 60:.1f} min", flush=True)
    return log_path

BASE_ENV = {
    "version": "v2",
    "_CUDA_VISIBLE_DEVICES": "0",
    "is_half": "True",
}

train_list_hits = glob.glob("/kaggle/input/**/train.list", recursive=True)
assert len(train_list_hits) == 1, train_list_hits
EN_DIR = os.path.dirname(train_list_hits[0])
for zip_path in glob.glob(f"{EN_DIR}/**/wav.zip", recursive=True):
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(os.path.dirname(zip_path))
wavs = sorted(glob.glob(f"{EN_DIR}/**/wav/*.wav", recursive=True))
assert len(wavs) == 251, f"expected 251 training wavs, got {len(wavs)}"
rows = pathlib.Path(train_list_hits[0]).read_text(encoding="utf-8").strip().splitlines()
assert len(rows) == 251, len(rows)
expected_names = set()
staged = []
for index, row in enumerate(rows, 1):
    parts = row.split("|", 3)
    assert len(parts) == 4, (index, row)
    wav_name, speaker, language, text = parts
    basename = os.path.basename(wav_name)
    assert language.upper() == "EN", (index, language)
    assert text.strip(), (index, row)
    assert basename not in expected_names, basename
    expected_names.add(basename)
    source = os.path.join(EN_DIR, "wav", basename)
    assert os.path.isfile(source), source
    target = os.path.join(DATA, basename)
    shutil.copy2(source, target)
    staged.append(f"{target}|{speaker}|en|{text}")
assert {os.path.basename(path) for path in wavs} == expected_names
LIST_PATH = f"{DATA}/mochi.list"
pathlib.Path(LIST_PATH).write_text("\n".join(staged) + "\n", encoding="utf-8")

REF_HITS = glob.glob("/kaggle/input/**/mochikosan-relax-003*.wav", recursive=True)
assert len(REF_HITS) == 1, REF_HITS
REF_WAV = REF_HITS[0]
with wave.open(REF_WAV, "rb") as ref_info:
    ref_duration = ref_info.getnframes() / ref_info.getframerate()
    assert ref_info.getnchannels() == 1
assert 3.0 <= ref_duration <= 10.0, ref_duration
print(f"staged {len(staged)} clips; reference={REF_WAV} ({ref_duration:.3f}s)", flush=True)

In [ ]:
# 4. Text/phoneme preprocessing with the repository import path explicitly exposed.
import glob
import os
import pathlib
import shutil
import sys

env = dict(BASE_ENV)
env.update({
    "inp_text": LIST_PATH,
    "inp_wav_dir": DATA,
    "exp_name": EXP,
    "opt_dir": OPT,
    "bert_pretrained_dir": f"{PRETRAINED}/chinese-roberta-wwm-ext-large",
    "i_part": "0",
    "all_parts": "1",
})
run_logged(
    "01_get_text",
    [sys.executable, "-u", "-s", f"{GSV_ROOT}/prepare_datasets/1-get-text.py"],
    env,
    timeout=1800,
)
part = f"{OPT}/2-name2text-0.txt"
merged = f"{OPT}/2-name2text.txt"
assert os.path.isfile(part), part
if os.path.exists(merged):
    os.remove(merged)
shutil.move(part, merged)
text_names = set()
for line in pathlib.Path(merged).read_text(encoding="utf-8").strip().splitlines():
    fields = line.split("\t")
    assert len(fields) == 4, line
    text_names.add(fields[0])
assert text_names == expected_names, (len(text_names), sorted(expected_names - text_names)[:5])
print(f"1a complete: {len(text_names)} phoneme entries", flush=True)

In [ ]:
# 5. HuBERT features and 32 kHz audio.
import glob
import os
import sys

env = dict(BASE_ENV)
env.update({
    "inp_text": LIST_PATH,
    "inp_wav_dir": DATA,
    "exp_name": EXP,
    "opt_dir": OPT,
    "cnhubert_base_dir": f"{PRETRAINED}/chinese-hubert-base",
    "i_part": "0",
    "all_parts": "1",
})
run_logged(
    "02_get_hubert",
    [sys.executable, "-u", "-s", f"{GSV_ROOT}/prepare_datasets/2-get-hubert-wav32k.py"],
    env,
    timeout=1800,
)
feature_names = {os.path.basename(x)[:-3] for x in glob.glob(f"{OPT}/4-cnhubert/*.pt")}
wav32_names = {os.path.basename(x) for x in glob.glob(f"{OPT}/5-wav32k/*.wav")}
assert feature_names == expected_names, (len(feature_names), sorted(expected_names - feature_names)[:5])
assert wav32_names == expected_names, (len(wav32_names), sorted(expected_names - wav32_names)[:5])
print(f"1b complete: {len(feature_names)} HuBERT features and wav32k files", flush=True)

In [ ]:
# 6. Semantic tokens.
import os
import pathlib
import shutil
import sys

env = dict(BASE_ENV)
env.update({
    "inp_text": LIST_PATH,
    "inp_wav_dir": DATA,
    "exp_name": EXP,
    "opt_dir": OPT,
    "pretrained_s2G": f"{PRETRAINED}/gsv-v2final-pretrained/s2G2333k.pth",
    "s2config_path": f"{GSV_ROOT}/configs/s2.json",
    "i_part": "0",
    "all_parts": "1",
})
run_logged(
    "03_get_semantic",
    [sys.executable, "-u", "-s", f"{GSV_ROOT}/prepare_datasets/3-get-semantic.py"],
    env,
    timeout=1800,
)
part = f"{OPT}/6-name2semantic-0.tsv"
semantic = f"{OPT}/6-name2semantic.tsv"
assert os.path.isfile(part), part
if os.path.exists(semantic):
    os.remove(semantic)
shutil.move(part, semantic)
semantic_names = set()
for line in pathlib.Path(semantic).read_text(encoding="utf-8").strip().splitlines():
    name, tokens = line.split("\t", 1)
    assert tokens.strip(), name
    semantic_names.add(name)
assert semantic_names == expected_names, (len(semantic_names), sorted(expected_names - semantic_names)[:5])
print(f"1c complete: {len(semantic_names)} semantic entries", flush=True)

In [ ]:
# 7. SoVITS configuration. The loader reads exp_dir folders directly; no training_files key.
import os

S2_WEIGHTS = f"{WORK}/stage_weights/SoVITS_weights_v2"
S2_LOG_DIR = f"{OPT}/logs_s2_v2"
S2_CONFIG_PATH = f"{TMP}/tmp_s2.json"
os.makedirs(S2_WEIGHTS, exist_ok=True)
os.makedirs(S2_LOG_DIR, exist_ok=True)
cfg = json.load(open(f"{GSV_ROOT}/configs/s2.json", encoding="utf-8"))
assert "training_files" not in cfg["data"]
cfg["train"].update({
    "batch_size": 8,
    "epochs": 8,
    "text_low_lr_rate": 0.4,
    "if_save_latest": True,
    "if_save_every_weights": True,
    "save_every_epoch": 4,
    "pretrained_s2G": f"{PRETRAINED}/gsv-v2final-pretrained/s2G2333k.pth",
    "pretrained_s2D": f"{PRETRAINED}/gsv-v2final-pretrained/s2D2333k.pth",
    "gpu_numbers": "0",
    "grad_ckpt": True,
})
cfg["model"]["version"] = "v2"
cfg["data"]["exp_dir"] = OPT
cfg["s2_ckpt_dir"] = S2_LOG_DIR
cfg["save_weight_dir"] = S2_WEIGHTS
cfg["name"] = EXP
cfg["version"] = "v2"
json.dump(cfg, open(S2_CONFIG_PATH, "w", encoding="utf-8"), indent=2)
print(f"s2 config ready: {S2_CONFIG_PATH}", flush=True)

In [ ]:
# 8. SoVITS training and checkpoint contract validation.
import glob
import os
import sys
import torch

run_logged(
    "04_s2_train",
    [sys.executable, "-u", "-s", f"{GSV_ROOT}/s2_train.py", "--config", S2_CONFIG_PATH],
    BASE_ENV,
    timeout=4 * 3600,
)
s2_candidates = sorted(glob.glob(f"{S2_WEIGHTS}/*.pth"), key=os.path.getmtime)
assert s2_candidates, f"no SoVITS exports in {S2_WEIGHTS}"
assert any("e8" in os.path.basename(path) for path in s2_candidates), s2_candidates
SOVITS_WEIGHT = s2_candidates[-1]
s2_payload = torch.load(SOVITS_WEIGHT, map_location="cpu", weights_only=False)
assert isinstance(s2_payload.get("weight"), dict)
assert "config" in s2_payload
assert s2_payload["config"]["model"]["version"] == "v2"
print(f"SoVITS export validated: {SOVITS_WEIGHT}", flush=True)

In [ ]:
# 9. GPT configuration.
import os
import yaml

GPT_WEIGHTS = f"{WORK}/stage_weights/GPT_weights_v2"
S1_LOG_DIR = f"{OPT}/logs_s1_v2"
S1_CONFIG_PATH = f"{TMP}/tmp_s1.yaml"
os.makedirs(GPT_WEIGHTS, exist_ok=True)
os.makedirs(S1_LOG_DIR, exist_ok=True)
cfg = yaml.safe_load(open(f"{GSV_ROOT}/configs/s1longer-v2.yaml", encoding="utf-8"))
cfg["train"].update({
    "batch_size": 6,
    "epochs": 8,
    "save_every_n_epoch": 4,
    "if_save_every_weights": True,
    "if_save_latest": True,
    "if_dpo": False,
    "half_weights_save_dir": GPT_WEIGHTS,
    "exp_name": EXP,
    "precision": "16-mixed",
})
cfg["data"]["max_sec"] = 12
cfg["pretrained_s1"] = f"{PRETRAINED}/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt"
cfg["train_semantic_path"] = f"{OPT}/6-name2semantic.tsv"
cfg["train_phoneme_path"] = f"{OPT}/2-name2text.txt"
cfg["output_dir"] = S1_LOG_DIR
yaml.safe_dump(cfg, open(S1_CONFIG_PATH, "w", encoding="utf-8"), sort_keys=False)
assert cfg["train"]["precision"] == "16-mixed"
print(f"s1 config ready: {S1_CONFIG_PATH}", flush=True)

In [ ]:
# 10. GPT training and checkpoint contract validation.
import glob
import os
import sys
import torch

s1_env = dict(BASE_ENV)
s1_env["hz"] = "25hz"
run_logged(
    "05_s1_train",
    [sys.executable, "-u", "-s", f"{GSV_ROOT}/s1_train.py", "--config_file", S1_CONFIG_PATH],
    s1_env,
    timeout=3 * 3600,
)
gpt_candidates = sorted(glob.glob(f"{GPT_WEIGHTS}/*.ckpt"), key=os.path.getmtime)
assert gpt_candidates, f"no GPT exports in {GPT_WEIGHTS}"
assert any("-e8.ckpt" in os.path.basename(path) for path in gpt_candidates), gpt_candidates
GPT_WEIGHT = gpt_candidates[-1]
gpt_payload = torch.load(GPT_WEIGHT, map_location="cpu", weights_only=False)
assert isinstance(gpt_payload.get("weight"), dict)
assert "config" in gpt_payload
assert gpt_payload["config"]["train"]["precision"] == "16-mixed"
print(f"GPT export validated: {GPT_WEIGHT}", flush=True)

In [ ]:
# 11. Package validated weights and reference before importing inference code.
import os
import pathlib
import shutil

MODEL_PARTIAL = f"{WORK}/mochi_model.partial"
if os.path.exists(MODEL_PARTIAL):
    shutil.rmtree(MODEL_PARTIAL)
os.makedirs(MODEL_PARTIAL, exist_ok=True)
shutil.copy2(SOVITS_WEIGHT, f"{MODEL_PARTIAL}/sovits_mochi_v2.pth")
shutil.copy2(GPT_WEIGHT, f"{MODEL_PARTIAL}/gpt_mochi_v2.ckpt")
shutil.copy2(REF_WAV, f"{MODEL_PARTIAL}/ref_mochiko.wav")
pathlib.Path(f"{MODEL_PARTIAL}/ref_text.txt").write_text(
    "そちらの商品は、お一人様につき一つまでとさせていただいております。\n",
    encoding="utf-8",
)
pathlib.Path(f"{MODEL_PARTIAL}/README.md").write_text(
    "# MochiVoice model v1 (GPT-SoVITS v2)\n\n"
    "Trained on mochi-train-en-v2 (251 F5-TTS English clips, Mochikosan timbre).\n\n"
    "- `sovits_mochi_v2.pth` — SoVITS v2 weights.\n"
    "- `gpt_mochi_v2.ckpt` — GPT weights.\n"
    "- `ref_mochiko.wav` and `ref_text.txt` — inference reference.\n"
    "- `test_en.wav` — deterministic English validation sample.\n\n"
    f"GPT-SoVITS commit: `{UPSTREAM_COMMIT}`\n"
    f"Hugging Face revision: `{HF_REVISION}`\n",
    encoding="utf-8",
)
print(f"protected package staged: {MODEL_PARTIAL}", flush=True)

In [ ]:
# 12. Inference validation against the fine-tuned weights.
import os
import pathlib
import sys
import numpy as np
import soundfile as sf

sys.path.insert(0, GSV_ROOT)
sys.path.insert(0, REPO)
os.environ.update({
    "PYTHONPATH": os.pathsep.join([GSV_ROOT, REPO, os.environ.get("PYTHONPATH", "")]),
    "gpt_path": f"{MODEL_PARTIAL}/gpt_mochi_v2.ckpt",
    "sovits_path": f"{MODEL_PARTIAL}/sovits_mochi_v2.pth",
    "cnhubert_base_path": f"{PRETRAINED}/chinese-hubert-base",
    "bert_path": f"{PRETRAINED}/chinese-roberta-wwm-ext-large",
    "version": "v2",
    "is_half": "True",
    "_CUDA_VISIBLE_DEVICES": "0",
    "language": "en_US",
})
from GPT_SoVITS import inference_webui as infer

# change_sovits_weights is a generator; consuming it is required to load the model.
infer.change_gpt_weights(os.environ["gpt_path"])
list(infer.change_sovits_weights(os.environ["sovits_path"]))
assert os.path.abspath(infer.gpt_path_global) == os.path.abspath(os.environ["gpt_path"])
prompt_label = next(key for key, value in infer.dict_language.items() if value == "all_ja")
target_label = next(key for key, value in infer.dict_language.items() if value == "en")
infer.set_seed(1234)
generated = list(infer.get_tts_wav(
    ref_wav_path=REF_WAV,
    prompt_text="そちらの商品は、お一人様につき一つまでとさせていただいております。",
    prompt_language=prompt_label,
    text="Hello, I really love the way you think about things.",
    text_language=target_label,
    top_k=15,
    top_p=1.0,
    temperature=1.0,
))
assert generated, "inference returned no audio"
sample_rate, audio = generated[-1]
samples = np.asarray(audio).squeeze()
assert sample_rate == 32000, sample_rate
assert samples.dtype == np.int16, samples.dtype
assert samples.ndim == 1 and samples.size > 0
normalized = samples.astype(np.float32) / 32767.0
duration = samples.size / sample_rate
assert 0.5 <= duration <= 30.0, duration
assert np.isfinite(normalized).all()
assert float(np.max(np.abs(normalized))) > 0.02, "silent inference output"
assert float(np.mean(np.abs(normalized))) > 0.002, "near-silent inference output"
assert float(np.mean(np.abs(samples) >= 32767)) < 0.01, "excessive clipping"
sf.write(f"{MODEL_PARTIAL}/test_en.wav", samples, sample_rate, subtype="PCM_16")
print(f"test_en.wav validated: {duration:.2f}s @ {sample_rate}Hz", flush=True)

In [ ]:
# 13. Finalize atomically and emit a reproducibility manifest outside the model directory.
import json
import os
import pathlib
import shutil
import soundfile as sf

expected_files = {
    "sovits_mochi_v2.pth",
    "gpt_mochi_v2.ckpt",
    "ref_mochiko.wav",
    "ref_text.txt",
    "test_en.wav",
    "README.md",
}
partial_files = {p.name for p in pathlib.Path(MODEL_PARTIAL).iterdir() if p.is_file()}
assert partial_files == expected_files, partial_files
final_dir = f"{WORK}/mochi_model"
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)
os.replace(MODEL_PARTIAL, final_dir)
final_files = {p.name for p in pathlib.Path(final_dir).iterdir() if p.is_file()}
assert final_files == expected_files, final_files
info = sf.info(f"{final_dir}/test_en.wav")
assert info.samplerate == 32000 and info.subtype == "PCM_16"
hashes = {}
for path in sorted(pathlib.Path(final_dir).iterdir()):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    hashes[path.name] = {"sha256": digest, "bytes": path.stat().st_size}
pathlib.Path(f"{WORK}/run_manifest.json").write_text(json.dumps({
    "status": "COMPLETE",
    "upstream_commit": UPSTREAM_COMMIT,
    "hf_revision": HF_REVISION,
    "files": hashes,
}, indent=2) + "\n", encoding="utf-8")
print(f"MODEL COMPLETE: {final_dir}", flush=True)
print(json.dumps(hashes, indent=2), flush=True)